# 7.2 Graph properties — every number is a claim about a rule

[07.1](07.1-social_graphs.ipynb) built the same messages into two networks and stopped at
the picture. This notebook is about the numbers you are allowed to put next to that picture,
and about one fact that runs through all of them: **a centrality score is computed from the
edges, and the edges were your decision.** Change the rule and the score changes, so the
score is a statement about the rule at least as much as about the person it is attached to.

The order is deliberate. First a graph small enough to read by eye, where the edges are
documented and the answer is known from outside the data — that is the calibration, the
place where you find out what degree and betweenness *mean* when they behave. Then the IRC
chat graph, which is four hundred nodes, has no documented edges at all, and where the same
three measures behave rather worse.

Barabási's [*Network Science*](http://networksciencebook.com/) chapter
[2](http://networksciencebook.com/chapter/2) is the reference for everything here: degree and
the degree distribution in 2.3, the adjacency matrix in 2.4, weighted networks in 2.6, paths
and distances in 2.8, connectedness in 2.9, the clustering coefficient in 2.10. It is free
and it is short.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from goad_toolkit.visualizer import HeatmapPlot, PlotSettings, ScatterPlot
from scipy.stats import pearsonr, spearmanr

from scripts.graphs import GraphPlot, MentionEdges, giant_component, induced, intransitivity, to_graph
from scripts.pipelines import BuildTimestamp, build_irc_pipeline
from scripts.plots import BarPlot
from wa_analyzer.data import load_showcase
from wa_analyzer.network_analysis import Config, GraphBuilder

## 7.2.1 A graph where the answer is known

Fifteen Florentine families in the early 1400s, an edge for a marriage between them. It is a
famous dataset because the answer is known from the historical record and not from the graph:
the Medici ended up running Florence, and the question is whether the marriage network saw it
coming.

Three measures, all of them one line of `networkx`:

- **Degree** — how many families you married into. $d_u = \sum_{v} A[u,v]$, the row sum of the
  adjacency matrix.
- **Betweenness** — the share of all shortest paths between other pairs that run through you.
  $c_b(v) = \sum_{s,t} \frac{\sigma(s,t \mid v)}{\sigma(s,t)}$, where $\sigma(s,t)$ counts the
  shortest paths from $s$ to $t$ and $\sigma(s,t \mid v)$ counts those that pass through $v$.
- **Clustering coefficient** — of all the pairs of your neighbours, what share are connected to
  each other. High means your contacts already know each other.

The clustering coefficient counts closed triangles. If you are friends with two people, are
they friends with each other? Sociologists call the tendency of such open triangles to close
**triadic closure**, and how closed the triangles around one person are can say something
about that person that no question about them does.

Bearman and Moody (2004, *American Journal of Public Health* 94(1), 89–95) had 13,465
American high-school students name their friends, and measured for each student the share
of their friends' friends who were not their own friends: an open friendship circle, made of
people who do not know each other. For girls, going from a fully closed circle to a fully
open one multiplied the odds of having seriously thought about suicide in the past year by
2.2 — with depression, self-esteem and family circumstances already in the model. For boys
there was no such effect. A number computed from nothing but who named whom picked up
something about these girls' lives that the questions about their mood did not.

That is the bar for a graph measure: **it should tell you something the columns you
already had do not.** The rest of this notebook keeps asking whether it does.

In [ ]:
florence = nx.florentine_families_graph()
families = pd.DataFrame(
    {
        "degree": dict(florence.degree()),
        "betweenness": nx.betweenness_centrality(florence),
        "clustering": nx.clustering(florence),
    }
).sort_values("betweenness", ascending=False)
print(families.round(3).to_string())
print(f"\n{florence.number_of_nodes()} families, {florence.number_of_edges()} marriages, "
      f"density {nx.density(florence):.2f}, diameter {nx.diameter(florence)}")

Read the top three rows against each other, because that is where the three measures stop
agreeing.

**The Medici have both the highest degree (6) and the highest betweenness (0.52), and those are
not the same achievement.** Their betweenness is twice the next family's, on a graph where the
most-married family has only six ties: half of every shortest path in Florence runs through
them. Their clustering coefficient is 0.07, which on six neighbours means exactly one of the
fifteen pairs of families they married into had also married each other. Fourteen of those
fifteen connections exist only by going back through the Medici.

**Strozzi and Guadagni both have degree 4.** By the measure a reader is most likely to be shown
they are equals. Their betweenness differs by a factor of two and a half — 0.26 against 0.10 —
because the Guadagni sit between parts of the network that have no other route to each other
(clustering 0.00: none of their four in-laws are connected), and the Strozzi married into a
corner that was already well connected (clustering 0.33). Peruzzi are the extreme case: degree 3,
clustering 0.67, betweenness 0.02. They are inside a clique, and a clique needs no broker.

This is what these measures are for. Degree counts your contacts; betweenness asks whether
anything would break if you left; the clustering coefficient asks whether your contacts need you
in order to reach each other. And the reason we are entitled to trust that reading is that the
answer can be checked against something outside the graph — the historical record of who ended
up governing Florence. Nothing below has that.

In [ ]:
top = families.head(8).reset_index(names="family")
bars = PlotSettings(figsize=(11, 4), title="Same families, three orderings", max_cols=3,
                    subplot_titles=["degree", "betweenness", "clustering coefficient"], ylabel="")
host = BarPlot(bars)
fig, axes = host.create_figure(n_plots=3)
for ax, column in zip(axes, ["degree", "betweenness", "clustering"]):
    colours = ["crimson" if f == "Medici" else "lightgrey" for f in top.sort_values(column, ascending=False).family]
    host.plot_on_axes(BarPlot(bars), ax, data=top.sort_values(column, ascending=False),
                      x=column, y="family", palette=colours, hue="family", legend=False)
    ax.set_ylabel("")

## 7.2.2 The same three measures on a chat

Now the IRC showcase, through 07.1's two rules on the same 2015 messages from `#ubuntu-uk`:
the **nearby** rule (an edge when two people spoke within ten minutes) and the **mention**
rule (an edge when one typed the other's nick).

`MentionEdges` is 07.1's `groupby` with its `isin(nicks)` guard, moved into `scripts/graphs.py`
so no graph gets built without it — a third of what the `addressed_to` regex catches is words
like `yeah,` rather than people.

In [ ]:
pipeline = build_irc_pipeline()
pipeline.add(BuildTimestamp)
irc = pipeline.apply(load_showcase("ubuntu_irc"))
chat = irc[(irc.channel == "#ubuntu-uk") & (irc.date.dt.year == 2015)].copy()

builder = GraphBuilder(Config(time_col="timestamp", node_col="author", seconds=600, datafile=None))
nearby = builder.build(chat, edge_seconds=600)
named = to_graph(MentionEdges()(chat), nodes=chat.author.unique())

for label, graph in [("nearby", nearby), ("named", named)]:
    components = sorted(nx.connected_components(graph), key=len, reverse=True)
    isolated = sum(1 for node in graph.nodes() if graph.degree(node) == 0)
    print(f"{label:7s} {graph.number_of_nodes()} nodes, {graph.number_of_edges():,} edges, "
          f"density {nx.density(graph):.4f}, {len(components)} components "
          f"(largest {len(components[0])}), {isolated} isolated")

**The component count is the first honest number here, and it is not in the picture.** The
mention graph is 188 pieces: one lump of 256 people and 187 nicks that nobody ever addressed
and who never addressed anybody. They are in the channel — they sent messages, they are nodes —
and the rule connects them to nothing. The nearby rule leaves only 39 such people, because
turning up at a busy moment is easier than being named.

Every measure from here on is computed on the largest component, which is the usual thing to do
and which quietly drops 42% of the people in one graph and 9% in the other. That belongs in the
caption too.

In [ ]:
core = giant_component(nearby)
core_named = giant_component(named)
print(f"nearby giant: {core.number_of_nodes()} nodes, average clustering "
      f"{nx.average_clustering(core):.3f}, diameter {nx.diameter(core)}, "
      f"average shortest path {nx.average_shortest_path_length(core):.2f}")
print(f"named giant:  {core_named.number_of_nodes()} nodes, average clustering "
      f"{nx.average_clustering(core_named):.3f}, diameter {nx.diameter(core_named)}, "
      f"average shortest path {nx.average_shortest_path_length(core_named):.2f}")

Both graphs have diameter 5 and an average path just over two: from anyone to anyone in about
two steps. That sounds like a finding about a tight-knit community and it is nothing of the
kind — a graph this dense, built from a rule this generous, could hardly come out otherwise.
The number to be surprised by would have been a *large* diameter.

In [ ]:
scores = pd.DataFrame(
    {
        "degree": dict(core.degree()),
        "weighted_degree": dict(core.degree(weight="weight")),
        "betweenness": nx.betweenness_centrality(core),
        "clustering": nx.clustering(core),
    }
)
scores["messages"] = chat.author.value_counts().reindex(scores.index)
scores["named_degree"] = pd.Series(dict(core_named.degree())).reindex(scores.index).fillna(0)
print("top 10 by degree under the nearby rule:")
print(scores.nlargest(10, "degree").round(3).to_string())

## 7.2.3 The bot problem is a rule problem

`lubotu3\`` is the channel's bug-tracker bot: it posts a link whenever somebody types a bug
number. It sends very few messages and it is present at every busy moment, which is exactly
what the nearby rule rewards.

In [ ]:
messages = chat.author.value_counts()
rank = scores.degree.rank(ascending=False, method="min")
peers = messages[(messages >= 25) & (messages <= 50)].index
peer_degree = scores.degree.reindex(peers).dropna()
reach = scores[scores.degree >= scores.degree.get("lubotu3`", 0)].messages

print(f"lubotu3`: {messages['lubotu3`']} messages, degree {scores.degree['lubotu3`']:.0f}, "
      f"ranked {rank['lubotu3`']:.0f} of {len(scores)} for degree")
print(f"humans who sent 25-50 messages: median degree {peer_degree.median():.0f}, "
      f"highest {peer_degree.max():.0f}")
print(f"everyone else at that degree or above sent a median of {reach.median():,.0f} messages")

**Thirty-six messages buys the thirty-sixth-highest degree of the 404 accounts the rule connects
to anybody.** The other accounts up at that level sent a median of 1,350 messages each. Nobody
ever addressed the bot by name, so under the mention rule it has degree 1 and sits near the
bottom — 07.1 already showed that pair of numbers.

The useful framing is not "drop the bots". It is that *the nearby rule measures presence*, and
a script that is always present scores highly on it because that is what it was asked to
measure. A bot is only the most obvious account for which "was in the room" and "took part" come
apart; a person who leaves an IRC client open all day is the same case with a less obvious name.
You will not find those by grepping for `bot`.

## 7.2.4 Weighted degree, and a correlation worth checking

`GraphBuilder` stores a weight on every edge: how many times the pair fell inside the same
window. Summing those is *weighted degree*, and it is the natural next measure to reach for
(Barabási 2.6). Before reading anything into it, check it against the column you already had.

In [ ]:
settings = PlotSettings(
    figsize=(7, 6),
    title="Weighted degree against the message count",
    xlabel="messages sent in 2015",
    ylabel="weighted degree, nearby rule",
)
paired = scores.dropna(subset=["messages"])
fig, ax = ScatterPlot(settings).plot(data=paired, x="messages", y="weighted_degree", color="#cccccc")
loud = paired.nlargest(4, "messages")
ax.scatter(loud.messages, loud.weighted_degree, color="#c44e52", zorder=3)
for row in loud.itertuples():
    ax.annotate(f"  {row.Index}", (row.messages, row.weighted_degree), color="#c44e52", va="center")
ax.set_xscale("log")
ax.set_yscale("log")

The scatter is a straight band across four orders of magnitude, and it should raise one doubt.
A handful of accounts sent thousands of messages each, and an ordinary (Pearson) correlation
weighs every point by how far it lies from the mean. **The band could be carried by the loudest
few accounts**, with the crowd of people who wrote twenty messages doing something else. What
you want to know is whether the crowd lines up too.

Ranks give every person one vote. A **spearman correlation** replaces every value by its rank —
1 for the most messages, 2 for the next — and computes the ordinary correlation on those ranks.
Eleven thousand messages buy the top rank and nothing more. So it answers exactly one question:
**do the two columns put people in the same order, all the way down?** +1 is the same order, 0 is
unrelated, −1 is the reverse.

It is also the natural question for a centrality: a centrality is only a ranking within one
graph (§7.2.6 shows why), so comparing rankings is comparing the thing it is. What it does not
tell you is *who* breaks the order. That takes a look at the ranks themselves, a little further down.

In [ ]:
for column in ["degree", "weighted_degree", "betweenness", "clustering"]:
    rho = spearmanr(paired[column], paired.messages).statistic
    print(f"spearman({column:16s}, messages sent) = {rho:+.2f}")
print(f"\nspearman(betweenness, degree)          = "
      f"{spearmanr(paired.betweenness, paired.degree).statistic:+.2f}")

**Weighted degree correlates 0.92 with the message count, on the ranks.** The order holds for the
crowd, not only for the top: the loudest accounts are not what makes the band. Be honest about how
much that adds, though. The log-log scatter already showed a straight band from one message to ten
thousand. The doubt was fair, and the answer is "no, and you could see it": ρ is the one-line
version of the scatter, the number you can put in a sentence.

What follows is the same either way. Weighted degree is the number of messages, run through a graph
and back out again. If a report leads with "X has the highest weighted degree in the network", a
reader who has the message counts already knows that, and the graph has added a layer of machinery
and no information. Say the message count; it is the honest version of the same sentence and nobody
has to trust your edge rule to believe it.

The check could have come out two other ways, and each would have sent you somewhere else.

- **Pearson high, ρ low.** The loud accounts lie on a line and the crowd does not. "Central means
  busy" then holds for the top few and for nobody else, and the next step is to look at the crowd
  without them.
- **ρ moderate, with people far from the diagonal in rank space.** Then the measure adds something,
  and the people off the diagonal are the finding. Plain degree is that case here: 0.76 with the
  message count. The next cell finds who leaves the diagonal.

In [ ]:
ranks = paired.rank(ascending=False, method="min")
gaps = pd.DataFrame({"messages": paired.messages, "degree": paired.degree,
                     "rank by messages": ranks.messages, "rank by degree": ranks.degree})
gaps["climbs"] = gaps["rank by messages"] - gaps["rank by degree"]
visitors = gaps[(gaps.messages <= 3) & (gaps.degree >= 5)]

print(gaps.loc[["lubotu3`"]].to_string())
print(f"\nmedian distance from the diagonal: {gaps.climbs.abs().median():.0f} places")
print("the eight biggest climbs, from rank by messages to rank by degree:")
print(gaps.nlargest(8, "climbs").to_string())
print(f"\n{len(visitors)} accounts with at most 3 messages and at least 5 neighbours")

`lubotu3\`` climbs seventy places, from 106th by messages to 36th by degree: §7.2.3's bot, now as a
point off the diagonal. It is not the biggest climber. Those are visitors: accounts with one to
three messages and five to ten neighbours, who climb from the bottom of the message ranking into
the middle of the degree ranking because their few messages landed in a busy ten minutes. Twenty-four
accounts sent at most three messages and still have five or more neighbours.

Those people are the finding, and it is §7.2.3's finding again: **the nearby rule measures
presence.** For a measure that adds something, this is what the check returns — not a verdict on
the measure, but a list of people to look at.

One more comparison shows where the gap between Pearson and spearman comes from: Pearson on the
values, Pearson on the logs, and spearman, side by side.

In [ ]:
x, y = paired.weighted_degree, paired.messages
print(f"pearson on the values: {pearsonr(x, y).statistic:.2f}")
print(f"pearson on the logs:   {pearsonr(np.log(x), np.log(y)).statistic:.2f}")
print(f"spearman:              {spearmanr(x, y).statistic:.2f}")
for k in (2, 10):
    rest = paired.drop(paired.nlargest(k, "messages").index)
    print(f"without the {k:2d} loudest accounts: pearson "
          f"{pearsonr(rest.weighted_degree, rest.messages).statistic:.2f}")

0.99, 0.94, 0.92. Taking logs pulls the tail in, and most of the distance between Pearson and
spearman goes with it: what separates the two is the tail. Here that distance is small. Without the
ten loudest accounts Pearson is still 0.96, so on this pair the loud accounts were never carrying
the band, which is what the scatter said.

Betweenness is barely better than weighted degree. It correlates 0.73 with messages sent and 0.80
with degree, so whatever it is picking up is mostly "this person is busy". Compare that with the
Florentine graph, where betweenness ranked Guadagni above Strozzi at equal degree and the ranking
meant something. The measure did not change. The graph did: the nearby rule connects everyone who
was awake at the same time, so there are no structural holes left for a broker to sit in.

**The check costs one line and it is not optional.** A centrality that reproduces a column you
already have is not a finding, and it is very easy to present as one.

## 7.2.5 One vote each, also for eighty identical people

Ranks gave every person one vote, and that protected §7.2.4 against a few extreme accounts. The
other side of it: one vote each does not protect against *many identical* people. The clustering
coefficient asks whether your neighbours know each other. Run it under both edge rules on the same
people and correlate it with degree.

In [ ]:
shared = sorted(set(core.nodes()) & set(core_named.nodes()))
both = pd.DataFrame(
    {
        "nearby_clustering": pd.Series(nx.clustering(core)).reindex(shared),
        "nearby_degree": pd.Series(dict(core.degree())).reindex(shared),
        "named_clustering": pd.Series(nx.clustering(core_named)).reindex(shared),
        "named_degree": pd.Series(dict(core_named.degree())).reindex(shared),
    }
)
print(f"{len(both)} people in both giant components")
print(f"average clustering, nearby rule: {both.nearby_clustering.mean():.3f}")
print(f"average clustering, mention rule: {both.named_clustering.mean():.3f}")
print(f"spearman(clustering, degree) under the nearby rule:  "
      f"{spearmanr(both.nearby_clustering, both.nearby_degree).statistic:+.2f}")
print(f"spearman(clustering, degree) under the mention rule: "
      f"{spearmanr(both.named_clustering, both.named_degree).statistic:+.2f}")

Same people, same year, same channel, and the sign flips. That reads like a finding — busy
people sit *between* groups under one rule and *inside* them under the other — and before
writing it down, look at what the two numbers summarise.

In [ ]:
pairs = PlotSettings(figsize=(11, 4.5), max_cols=2, title="Clustering against degree, the same people",
                     subplot_titles=["nearby rule", "mention rule"], xlabel="degree (log scale)",
                     ylabel="clustering coefficient")
host = ScatterPlot(pairs)
fig, axes = host.create_figure(n_plots=2)
for ax, rule in zip(axes, ["nearby", "named"]):
    degree, clustering = both[f"{rule}_degree"], both[f"{rule}_clustering"]
    alone = degree == 1
    host.plot_on_axes(ScatterPlot(pairs), ax, data=both, x=f"{rule}_degree", y=f"{rule}_clustering",
                      color="#cccccc")
    ax.scatter(degree[alone], clustering[alone], color="#c44e52", s=80, zorder=3)
    ax.annotate(f"  {alone.sum()} people with one neighbour", (1, 0), color="#c44e52", va="bottom")
    ax.set_xscale("log")
    print(f"{rule:6s} rule: spearman {spearmanr(degree, clustering).statistic:+.2f} with them, "
          f"{spearmanr(degree[~alone], clustering[~alone]).statistic:+.2f} without them")
    for label, x in [("pearson", degree), ("pearson on log degree", np.log(degree))]:
        print(f"    {label:22s} {pearsonr(x, clustering).statistic:+.2f} with them, "
              f"{pearsonr(x[~alone], clustering[~alone]).statistic:+.2f} without them")

**The picture shows one relation, twice.** Under both rules clustering falls as degree rises:
the people with the most contacts are the people whose contacts least often know each other.
Leave out the people with a single neighbour and both correlations are strongly negative.

The red dot is where the positive sign comes from. Someone with one neighbour has no pair of
neighbours, so their clustering coefficient is 0 by definition — `networkx` returns 0 for a
number that is really undefined. Under the mention rule 82 of the 255 people have exactly one
neighbour, so 82 people sit on the same point, (1, 0), and with one vote each that point gets 82
votes. Spearman reads the pile in the corner as "low degree goes with low clustering". It has no
way of knowing the pile is an artefact of the definition.

That is not a weakness of ranks alone. Pearson on log(degree) changes sign the same way, +0.33 with
the pile and −0.66 without it, and Pearson on raw degree goes from −0.66 to −0.06. Every correlation
in that cell is moved by the 82; the sign was decided by how many people had a single contact,
which is a fact about the rule, not about the hubs. **A correlation summarises a scatter; draw the scatter first.** And
when a measure is undefined for some nodes, say how many there are and leave them out, rather
than let a library fill in a zero.

**The same triangles, one step further out.** The clustering coefficient asks whether your
neighbours know each other. Bearman and Moody, in §7.2.1, looked one step further: of your
neighbours' neighbours, the share who are not your own neighbours. They called it the
*intransitivity index*, and every such person is a triangle around you that did not close. It is
not in `networkx`; `intransitivity` in `scripts/graphs.py` computes it in a few lines.

For the girls in their study it added something that depression and self-esteem did not. Before
borrowing it, ask it this notebook's question: does it add something to a column we already have?
Degree is the obvious suspect, because it counts the same neighbours.

In [ ]:
pairs = PlotSettings(figsize=(11, 4.5), max_cols=2, title="Intransitivity against degree, the same chat",
                     subplot_titles=["nearby rule", "mention rule"], xlabel="degree (log scale)",
                     ylabel="intransitivity")
host = ScatterPlot(pairs)
fig, axes = host.create_figure(n_plots=2)
for ax, (rule, graph) in zip(axes, [("nearby", core), ("named", core_named)]):
    measures = pd.DataFrame({"intransitivity": intransitivity(graph), "clustering": nx.clustering(graph),
                             "degree": dict(graph.degree())})
    alone = measures.degree == 1
    host.plot_on_axes(ScatterPlot(pairs), ax, data=measures, x="degree", y="intransitivity",
                      color="#cccccc")
    ax.scatter(measures.degree[alone], measures.intransitivity[alone], color="#c44e52", s=80, zorder=3)
    ax.set_xscale("log")
    rest = measures[~alone]
    similar = rest[rest.degree.between(5, 15)]
    print(f"{rule:6s} rule: {alone.sum()} people with one neighbour, intransitivity "
          f"{sorted(set(measures.intransitivity[alone]))}")
    print(f"    without them, spearman with degree {spearmanr(rest.intransitivity, rest.degree).statistic:+.2f}, "
          f"with clustering {spearmanr(rest.intransitivity, rest.clustering).statistic:+.2f}")
    print(f"    {len(similar)} people with 5 to 15 neighbours: intransitivity "
          f"{similar.intransitivity.min():.2f} to {similar.intransitivity.max():.2f}")
    print(f"    the three highest degrees: {rest.nlargest(3, 'degree').intransitivity.round(2).to_dict()}")

**On this chat, intransitivity is degree upside down.** Spearman −0.98 under both rules, and each
panel is one falling curve. That is the shape of the network, not anything about the people in it. A
hub is adjacent to nearly everyone, so its neighbours' neighbours are mostly its own neighbours:
popey, with 215 neighbours, scores 0.12 under the nearby rule. Someone with one neighbour scores
exactly 1 by definition, because their only contact's other contacts cannot be theirs. The red dot
is §7.2.5's pile again, this time at the top. And among people with the same number of neighbours
there is almost nothing left to measure: with five to fifteen neighbours, everyone sits between 0.89
and 0.98 under the nearby rule. Intransitivity even rises with clustering (+0.76), only because both
fall with degree.

So why did it work for Bearman and Moody? Add Health asked each student to name at most five female
and five male friends, and the two steps in their measure are two such nominations. Degree could not
run from 1 to 215, so it could not swamp the measure, and what was left to vary was the shape of the
circle. Here nothing caps degree, and the measure reports degree again. **Same formula, different
data.** Before you borrow a measure from a paper, check what it reduces to on your graph: compare
people with the same number of neighbours, and see whether anything is left.

## 7.2.6 Why you cannot compare a centrality across graphs

Betweenness in `networkx` is normalised by default: divided by the number of pairs it could
possibly sit between, so it lands in $[0, 1]$ and looks comparable. It is not.

In [ ]:
rows = []
for label, graph in [("florentine", florence), ("karate club", nx.karate_club_graph()),
                     ("irc nearby giant", core)]:
    normalised = nx.betweenness_centrality(graph)
    raw = nx.betweenness_centrality(graph, normalized=False)
    rows.append({
        "graph": label, "nodes": graph.number_of_nodes(),
        "max normalised": max(normalised.values()), "mean normalised": np.mean(list(normalised.values())),
        "max raw": max(raw.values()), "max degree": max(dict(graph.degree()).values()),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

The most central family in a fifteen-node network scores 0.52. The most central account in a
four-hundred-node network scores 0.18. On the raw scale the ranking reverses and the ratio is
three hundred to one. Both columns are correct arithmetic and neither supports the sentence
"the Medici were more central to Florence than daftykins was to `#ubuntu-uk`".

The reason is that normalising divides by the number of *pairs*, which grows with $n^2$, while
the paths a single node can actually capture do not. Maximum degree has the same problem in the
other direction: 6 against 215, and the second graph is not thirty-five times more social. **A
centrality is a ranking within one graph. Across graphs it is not a quantity.** If you need to
compare two networks, compare their rankings — is the top account also top under the other
rule — or compare each against a graph of the same size, which is what
[07.3](07.3-clusters-in-graphs.ipynb) does for modularity.

## 7.2.7 Position in a layout is not data

Degree, betweenness and clustering are properties of the graph. The picture is not: a layout
is a rule for placing nodes, and a different rule gives a different picture. A spring layout
treats edges as springs and nodes as repelling charges and settles wherever it settles; a
circular layout puts everybody on one ring. Two layouts of one graph, coloured and sized by
the same degrees.

In [ ]:
hubs = giant_component(to_graph(MentionEdges()(chat)))
hubs = giant_component(induced(hubs, [n for n in hubs.nodes() if hubs.degree(n) >= 6]))
degrees = np.array([hubs.degree(n) for n in hubs.nodes()], dtype=float)

layout = PlotSettings(figsize=(11, 5), max_cols=2, title="One graph, two layouts, identical degrees",
                      subplot_titles=["spring layout", "circular layout"])
host = GraphPlot(layout)
fig, axes = host.create_figure(n_plots=2)
for ax, pos in zip(axes, [nx.spring_layout(hubs, seed=1, k=0.35), nx.circular_layout(hubs)]):
    host.plot_on_axes(GraphPlot(layout), ax, data=hubs, pos=pos,
                      color=degrees, node_size=3 * degrees, cmap="viridis")
print(f"{hubs.number_of_nodes()} people addressed by at least six others; "
      f"highest degree {int(degrees.max())}, median {np.median(degrees):.0f}")

The two panels look nothing alike. The spring layout pulls the hubs into a bright knot in the
middle and pushes the people with few ties to the rim; the circular layout gives everyone the
same place on one ring, and the hubs are only visible through colour and size. Every degree,
every betweenness and every clustering coefficient is identical between them. So: **never
write that someone is "on the edge of the network"** when what happened is that this layout
put them there. Write their degree, which will still be true tomorrow with a different layout.

A drawing is one way to show a graph. The other is the **adjacency matrix** itself, as a
heatmap: one row and one column per person, a coloured cell where two people are connected.
It has no layout to misread, but it has an order, and the order is a choice too — here by
degree, so the hubs are the top rows.

In [ ]:
order = sorted(hubs.nodes(), key=hubs.degree, reverse=True)
matrix = pd.DataFrame(nx.to_numpy_array(hubs, nodelist=order, weight="weight"), index=order, columns=order)

heat = PlotSettings(figsize=(7, 6), title="The same graph as a matrix, people ordered by degree",
                    xlabel="", ylabel="")
fig, ax = HeatmapPlot(heat).plot(data=np.log1p(matrix), cmap="Blues", annot=False,
                                 xticklabels=False, yticklabels=False,
                                 cbar_kws={"label": "log(1 + messages addressed)"})

The matrix shows what the drawings hide: the weights. The dark cells sit almost all in the top
rows and left columns — most messages are between hubs, or from someone to a hub — and the
bottom right, where the people with six or seven ties meet each other, is nearly empty. In the
spring layout that same fact is the knot in the middle, without the weights.

Neither view is the better one in general. A drawing shows paths, bridges and who sits between
whom, and it fails once the graph is dense: a few hundred edges is a hairball. A matrix shows
weights and blocks and never becomes a hairball, but a path of three steps is invisible in it.
**Draw both, side by side, and use the one that shows your point.** With the rows ordered by a
clustering, as in [07.3](07.3-clusters-in-graphs.ipynb), the matrix is also the clearest
picture of a community there is.

The thing that *is* in the drawing is the degree, because we mapped it to colour and size on
purpose. That mapping is the caption's job.

**Where does the dark block end?** Your eye picks out a square in the top left of the matrix,
larger than the ten busiest people and smaller than half the graph. A line at "the top ten" would
be a choice you cannot defend. A **k-core** gives the line a definition: the largest group in which
everyone is tied to at least $k$ others *inside the group*. `networkx` finds it by peeling: remove
everyone with fewer than $k$ ties, which lowers the degree of their neighbours, remove whoever drops
below $k$ next, and repeat until nobody is left to remove. Whoever survives is the $k$-core. Each
person's core number is the highest $k$ they survive, and the people with the highest core number
form the main core.

In [ ]:
core_numbers = pd.Series(nx.core_number(hubs))
k = core_numbers.max()
size = (core_numbers == k).sum()
print(f"main core: k = {k}, {size} people; "
      f"they are the {size} highest degrees: {set(order[:size]) == set(core_numbers[core_numbers == k].index)}")
print(core_numbers.value_counts().sort_index(ascending=False).head(5).rename("people").to_string())

ties = nx.to_numpy_array(hubs, nodelist=order, weight=None)
rest = len(order) - size
print(f"\npairs that talk: inside the core {ties[:size, :size].sum() / (size * (size - 1)):.0%}, "
      f"core with the rest {ties[:size, size:].mean():.0%}, "
      f"rest among themselves {ties[size:, size:].sum() / (rest * (rest - 1)):.0%}")
print(f"mentions inside the core: {matrix.iloc[:size, :size].to_numpy().sum() / matrix.to_numpy().sum():.0%}")

marked = PlotSettings(figsize=(7, 6), title=f"The same matrix, the main {k}-core marked ({size} people)",
                      xlabel="", ylabel="")
fig, ax = HeatmapPlot(marked).plot(data=np.log1p(matrix), cmap="Blues", annot=False,
                                   xticklabels=False, yticklabels=False,
                                   cbar_kws={"label": "log(1 + messages addressed)"})
ax.axhline(size, color="crimson", lw=1.5)
ax.axvline(size, color="crimson", lw=1.5)

The main core has $k = 19$: 25 people, each tied to at least 19 of the other 24. They are exactly the
25 highest degrees, so the core is the top-left square and two lines mark it. Inside it 91% of the
pairs talk, its rows reach 35% of everyone else, and the rest talk among themselves in 5% of their
pairs. Nearly three quarters of all mentions stay inside the core. That is the hub network the
drawing hinted at, now with a border you can state in one sentence and someone else can reproduce:
*the main 19-core*.

Two things the line does not say. The core is a square here only because the rows are sorted by
degree; sort them another way and the same 25 people are scattered over the matrix, so check that
they are the top rows before you draw one line. And a k-core is not a set of communities: it peels
the graph down to its one densest centre and says nothing about groups inside it or around it. That
is the question of [07.3](07.3-clusters-in-graphs.ipynb).

## 7.2.8 The long tail arrives here too

One last number, and it is the reason a summary statistic of this graph describes nobody. Lesson
4's question — is this one population or several — applies to a degree distribution as much as to
a flipper length (Barabási 2.3, and chapter [4](http://networksciencebook.com/chapter/4) for what
a heavy tail does to a network).

In [ ]:
all_degrees = pd.Series(dict(nearby.degree()))
touched = {node for node, _ in all_degrees.nlargest(10).items()}
carried = sum(1 for u, v in nearby.edges() if u in touched or v in touched)
print(all_degrees.describe().round(1).to_string())
print(f"\nshare of accounts with degree <= 5: {(all_degrees <= 5).mean():.1%}")
print(f"share of edges touching the ten highest-degree accounts: "
      f"{carried:,}/{nearby.number_of_edges():,} = {carried / nearby.number_of_edges():.1%}")

Mean degree 14.7, median 5, maximum 215. Over half the channel has five neighbours or fewer, and
ten accounts out of 443 sit on nearly half the edges. "The average person in this network has
fifteen connections" is arithmetically true and describes nobody: not the 55% below five, and
certainly not the ten who hold the thing together.

## 7.2.9 Does the ranking survive the window?

Everything above used a ten-minute window, and 07.1 showed that the window decides how many edges
there are. Spearman has a second use here, different from §7.2.4. There it asked *does this measure
add something to a column I already had?* Here it asks *does the ranking hold if I change a choice I
made?* — the same measure, computed once under each choice. A ranking that survives is a property
of the people; one that does not is a property of the window. Lesson 5 used it this way in
[05.5](../lesson5/05.5-how-people-type.ipynb): the same habits, four years apart.

The cell rebuilds the graph at 1, 5, 10 and 30 minutes and compares each measure with its own
ten-minute ranking, on the people both giant components contain. Clustering is compared twice:
for everyone, and only for people with at least two neighbours in both windows, where it is
defined.

In [ ]:
windows = {}
for seconds in (60, 300, 600, 1800):
    giant = giant_component(builder.build(chat, edge_seconds=seconds))
    windows[seconds] = pd.DataFrame({
        "degree": dict(giant.degree()),
        "betweenness": nx.betweenness_centrality(giant),
        "clustering": nx.clustering(giant),
    })

rows = []
for seconds, measures in windows.items():
    common = measures.index.intersection(windows[600].index)
    a, b = measures.loc[common], windows[600].loc[common]
    defined = (a.degree >= 2) & (b.degree >= 2)
    rows.append({
        "window": f"{seconds // 60} min", "people": len(common),
        **{column: spearmanr(a[column], b[column]).statistic
           for column in ["degree", "betweenness", "clustering"]},
        "clustering, 2+ neighbours": spearmanr(a.clustering[defined], b.clustering[defined]).statistic,
    })
print(pd.DataFrame(rows).round(2).to_string(index=False))

ends = windows[60].index.intersection(windows[1800].index)
print("\n1 min against 30 min: " + ", ".join(
    f"{column} {spearmanr(windows[60].loc[ends, column], windows[1800].loc[ends, column]).statistic:.2f}"
    for column in ["degree", "betweenness", "clustering"]))

**Degree and betweenness hold.** Against the ten-minute ranking, degree keeps 0.92 at one minute and
0.95 at thirty, betweenness 0.90 at both. Betweenness is as stable as degree here, which fits
§7.2.4: in this graph it mostly tracks who is busy, and who is busy does not depend on the window.

**Clustering does not, at first sight: 0.45 at one minute.** Most of that is §7.2.5 again. A
one-minute window leaves more people with a single neighbour, and their clustering is 0 by
definition. Among people with at least two neighbours in both windows the ranking holds at 0.85
to 0.93. Between the two ends of the sweep, one minute against thirty, clustering for everyone
comes out at 0.29, against 0.87 for degree.

Had the dotted line dropped as well, the clustering ranking would have been a property of the
window, and the next step would be to show *who* moves: the ranking per window, side by side, and
the measure reported only together with its window. As it is, clustering can be reported with
§7.2.5's rule: leave out the people for whom it is undefined, and say how many.

A ranking of four hundred people can carry one summary number. A ranking of eight cannot, and
that is the first thing to know before you run this on your own chat:

In [ ]:
eight = chat.author.value_counts().nlargest(8).index
for seconds in (60, 1800):
    small = builder.build(chat[chat.author.isin(eight)], edge_seconds=seconds)
    print(f"the eight busiest accounts alone, {seconds // 60:2d} min: "
          f"{small.number_of_edges()} of 28 possible edges, "
          f"clustering values {sorted(set(nx.clustering(small).values()))}")
print(f"\nspread of spearman between two unrelated orderings of 8 people: 1/sqrt(7) = {1 / np.sqrt(7):.2f}")

## 7.2.10 Your turn

Once you have a graph, the question is not which measures `networkx` can compute — it can
compute dozens — but **which of them adds something to what you already knew about your
chat**. A centrality that reproduces the message count adds nothing; a clustering coefficient
that picks out the one person whose friends do not talk to each other might.

Your own chat is smaller and denser than an IRC channel — a group of eight is close to complete
at any window, so degree will separate almost nobody. That is the point of doing it: the measure
does not fail loudly, it just returns numbers that all look alike, and you have to notice.

```python
from scripts.graphs import giant_component
from wa_analyzer.data import load_own_chat
from wa_analyzer.network_analysis import Config, GraphBuilder

own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
builder = GraphBuilder(Config(time_col="timestamp", node_col="author", seconds=600, datafile=None))
graph = builder.build(own, edge_seconds=600)

scores = pd.DataFrame({
    "degree": dict(graph.degree()),
    "weighted_degree": dict(graph.degree(weight="weight")),
    "betweenness": nx.betweenness_centrality(giant_component(graph)),
    "clustering": nx.clustering(giant_component(graph)),
})
scores["messages"] = own.author.value_counts().reindex(scores.index)
print(spearmanr(scores.weighted_degree, scores.messages).statistic)

# assignment 2: one column per window, each person's rank in it
clustering = pd.DataFrame({f"{s}s": nx.clustering(builder.build(own, edge_seconds=s))
                           for s in (60, 1800)})
print(clustering.rank(ascending=False).sort_values("60s"))
```

> **Your turn.**
>
> 1. Run the correlation between weighted degree and message count on your own chat, as §7.2.4
>    did. If it is above 0.9, say so — you have found that your centrality is a message counter,
>    which is a result and saves you from writing a sentence you cannot defend.
> 2. Compute the clustering coefficient at a 60-second window and at a 30-minute window, as
>    §7.2.9 did, and leave out anyone with fewer than two neighbours. Do not summarise it with one
>    spearman: on eight people, two orderings with nothing to do with each other give a ρ with a
>    spread of $1/\sqrt{7} \approx 0.38$, so a ρ of 0.6 is less than two of those from zero. Show
>    the two rankings side by side instead, as the last lines of the code above do, and say who
>    moves. If the ordering changes, it is a property of the window. And check first that there
>    is something to rank: the eight busiest `#ubuntu-uk` accounts on their own are a complete
>    graph at both windows, and every clustering coefficient is 1.
> 3. Count the components and the isolated nodes. In a group chat there will be one component and
>    no isolates, which tells you that the *rule* cannot separate anybody in a chat this size —
>    not that everybody is equally close.
> 4. Find your `lubotu3\``: an account whose degree is high relative to how much it says. In a
>    WhatsApp export that is often the person who only ever reacts, or a number with no name.
> 5. Draw your graph and its adjacency matrix side by side. Which one shows you something the
>    other does not?
> 6. For every measure you keep, write one sentence on what it tells you about your chat that
>    the message counts did not. A measure without that sentence goes.

## 7.2.11 What to write down

1. **The edge rule, again**, and this time next to the measure. Not "degree" but "degree under a
   ten-minute window".
2. **Which component you measured on**, and how many people that dropped.
3. **One correlation between your centrality and a column you already had**, with the scatter
   it summarises. If it is high, that is your finding.
4. **Whether the ranking survives the window.** One ρ for a large group, the rankings side by side
   for a small one.
5. **A number, not a layout.** If your evidence for someone being central is where the spring
   layout put them, you have no evidence.
6. **Nothing compared across two graphs of different size** unless you compared rankings.

---

**Where this goes next.** Every measure in this notebook is about one node at a time. The next
question is about groups: does this network split into communities, and would you know if the
split were an artefact? [07.3-clusters-in-graphs](07.3-clusters-in-graphs.ipynb) does it on the
karate club, where the true split is recorded, and then on this chat graph, where it is not.